# Forget-MI BASELINE — Kaggle Notebook (EXP1-M)

> 🎯 **Mục đích**: Reproduce paper Forget-MI Table 2 trên Kaggle để có **số time/GPU đo trên cùng GPU với LoKU** (so sánh fair trong Chương 4 luận văn).
>
> **Support 2 datasets**: MIMIC-CXR (sẵn sàng) + Indiana University CXR (cần prep trước).

## Đặc điểm baseline vs LoKU
| | Baseline (notebook này) | LoKU (run.ipynb trên Colab) |
|---|---|---|
| Script | `training/forgetmi_partial.py` | `training/forgetmi_loku.py` |
| Trainable params | **100%** (full FT) | ~0.45% (LoRA + FILA) |
| Time/run | **~5h** (30 epochs) | ~12 min (8 epochs) |
| Loss | 4-loss Forget-MI gốc | 4-loss + IHL + FILA + distill |

## Cấu trúc notebook (15 cells)
| Cell | Mục đích | Phụ thuộc |
|---|---|---|
| 1 | Setup Kaggle env: clone repo, install deps | Mọi session |
| 2 | Verify input datasets + GPU (cả MIMIC + IU) | Sau Cell 1 |
| 3 | Define helpers + DATASETS config | **BẮT BUỘC trước Cell 4*** |
| **4a / 4b / 4c** | **MIMIC-CXR** — Train 3% / 6% / 10% (multi-seed) | Cần Kaggle Datasets MIMIC |
| **4d / 4e / 4f** | **IU-CXR** — Train 3% / 6% / 10% (multi-seed) | Cần IU prep xong + `enabled=True` |
| 5 | Bảng cross-dataset (MIMIC + IU × 3 forget% × Paper) | Sau Cell 4* |
| 6 | Push results lên GitHub (qua Kaggle Secrets) | Cần Kaggle Secrets |

## Setup Kaggle 1 lần (trước khi chạy)

### Bắt buộc (cho MIMIC)
1. **Kaggle Datasets** (Sidebar → + Add data → Upload):
   - `forget-mi-data` — chứa `metadata/` + `img_data/`
   - `forget-mi-models` — chứa `training_original_model/` + `model_retrained_3per/`
2. **Kaggle Secrets** (Add-ons → Secrets): `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME`
3. **GPU**: Settings → Accelerator → **GPU T4 x2** (free 30h/tuần) hoặc **P100** nếu Pro

### Bổ sung (cho IU — khi đã prep xong theo THESIS_ROADMAP Section 13)
4. Tạo `config_baseline_iu_kaggle.yaml` (copy từ `config_baseline_kaggle.yaml`, đổi paths IU + `output_channel_encoding=binary`)
5. **Kaggle Datasets** thêm:
   - `forget-mi-data-iu`
   - `forget-mi-models-iu` (có `model_og_IU` + 3 `model_retrained_iu_Nper`)
6. Trong Cell 3, set `DATASETS['iu']['enabled'] = True`, rerun Cell 3

## Quota Kaggle free: 30h GPU/tuần

| Dataset | Forget% | Time | Cộng dồn |
|---|---|---|---|
| MIMIC 3% | × 3 seeds | ~15h | 15h |
| MIMIC 6% | × 3 seeds | ~15h | 30h (vừa khít tuần) |
| MIMIC 10% | × 3 seeds | ~15h | 45h (sang tuần sau) |
| IU 3-6-10% | × 2 seeds | ~30h | thêm 1 tuần |

→ Tổng đầy đủ 2 datasets: **~3 tuần** với free quota. Khuyến nghị: chạy MIMIC trước (xong sớm cho luận văn), IU sau."""

In [ ]:
# ====================================
# CELL 1: Setup Kaggle env (clone repo + install deps + restore CSV)
# ====================================
import os, sys, subprocess, shutil

WORK_DIR = "/kaggle/working"
REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"
REPO_NAME = "Forget-MI-LoKU"
REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"

os.chdir(WORK_DIR)

# Auth: token từ Kaggle Secrets (repo có thể private)
def _get_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception as e:
        print(f"⚠️  Không load được GITHUB_TOKEN ({e}) — thử clone anonymous")
        return None

_token = _get_github_token()
_REPO_URL_AUTH = (f"https://{_token}@github.com/nhnhu146/Forget-MI-LoKU.git"
                  if _token else REPO_URL)

# Clone hoặc pull
if not os.path.exists(REPO_DIR):
    print(f"🔽 Clone {REPO_URL}")
    subprocess.run(["git", "clone", _REPO_URL_AUTH, REPO_DIR], check=True)
else:
    print(f"🔄 Pull latest")
    if _token:
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _REPO_URL_AUTH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/master"], check=True)

os.chdir(REPO_DIR)
print(f"📂 CWD: {os.getcwd()}")
subprocess.run(["git", "log", "--oneline", "-1"])

# Restore CSV từ repo (nếu Cell 6 trước đó đã push)
csv_in_repo = "experiments/results_summary_kaggle.csv"
csv_target = "/kaggle/working/results_summary.csv"
if os.path.exists(csv_in_repo) and not os.path.exists(csv_target):
    shutil.copy(csv_in_repo, csv_target)
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"♻️  Restored CSV: {csv_target} ({n} rows) → seed_done sẽ skip đúng")
elif os.path.exists(csv_target):
    n = sum(1 for _ in open(csv_target)) - 1
    print(f"📊 CSV hiện có: {csv_target} ({n} rows)")
else:
    print("📊 CSV chưa có — sweep sẽ chạy từ đầu")

# Install pinned ML stack (let pip auto-resolve sub-deps for transformers 4.38)
print("")
print("📦 Installing ML stack (pip auto-resolve subs)...")
# Extra utils (not in transformers dep tree)
subprocess.run(["pip", "install", "-q", "pydicom", "scikit-image", "pyyaml"], check=True)
# Main pin: transformers 4.38.0 — pip will downgrade tokenizers/huggingface-hub/
# safetensors to versions compatible (e.g. tokenizers<0.19, huggingface-hub<1.0).
# torch is kept (Kaggle preinstall satisfies torch>=1.13).
subprocess.run([
    "pip", "install", "-q",
    "transformers==4.38.0", "peft==0.10.0", "accelerate==0.27.0",
], check=True)
print("✅ ML stack installed")

# Check GPU
import torch
if torch.cuda.is_available():
    print("")
    print(f"🟢 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print("")
    print("🔴 KHÔNG CÓ GPU — baseline sẽ KHÔNG xong trong 12h Kaggle limit!")
    print("   → Settings → Accelerator → GPU T4 x2 → Save → Restart Session")

print("")
print("✅ Setup complete")

In [ ]:
# ====================================
# CELL 2: Verify input Kaggle datasets (AUTO-DETECT mount format)
# ====================================
# Kaggle có 2 format mount paths:
#   Cũ: /kaggle/input/<slug>/...
#   Mới: /kaggle/input/datasets/<username>/<slug>/...
# Cell này tự detect và verify từng dataset.

import os
import glob


def _find_kaggle_dataset(slug):
    """Find Kaggle dataset folder by slug. Return path hoặc None."""
    # Format CŨ
    direct = f'/kaggle/input/{slug}'
    if os.path.isdir(direct):
        return direct
    # Format MỚI
    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')
    if candidates:
        return candidates[0]
    return None


print("🔍 Auto-detect Kaggle dataset paths:\n")

# ---- Files chung (từ repo clone) ----
COMMON = {
    "Synonyms":     "./data_splits/Synonyms.csv",
    "Forget 3%":    "./data_splits/forget_set_3per.csv",
    "Forget 6%":    "./data_splits/forget_set_6per.csv",
    "Forget 10%":   "./data_splits/forget_set_10per.csv",
}

# Detect MIMIC datasets
mimic_data_root = _find_kaggle_dataset('forget-mi-data')
mimic_models_root = (_find_kaggle_dataset('forget-mi-models-full')
                     or _find_kaggle_dataset('forget-mi-models-v2')
                     or _find_kaggle_dataset('forget-mi-models'))

print(f"📦 forget-mi-data   → {mimic_data_root or '❌ NOT FOUND'}")
print(f"📦 forget-mi-models → {mimic_models_root or '❌ NOT FOUND'}")
print()

# Build MIMIC paths dynamically
if mimic_data_root and mimic_models_root:
    # Auto-pick paths based on dataset slug (full vs old)
    if 'forget-mi-models-full' in mimic_models_root:
        _base = f"{mimic_models_root}/original_model/forgetme/training_original_model"
        _re3  = f"{mimic_models_root}/model_retrained_3per/model_retrained_3per"
        _re6  = f"{mimic_models_root}/model_retrained_6per/model_retrained_6per"
        _re10 = f"{mimic_models_root}/model_retrained_10per/model_retrained_10per"
    else:
        _base = f"{mimic_models_root}/base_model/training_original_model"
        _re3  = f"{mimic_models_root}/retrained_model/model_retrained_3per"
        _re6  = None
        _re10 = None
    MIMIC = {
        "Base model":      f"{_base}/pytorch_model.bin",
        "Retrained 3%":    f"{_re3}/pytorch_model.bin",
        "Text metadata":   f"{mimic_data_root}/data/metadata",
        "Image data":      f"{mimic_data_root}/data/img_data",
        "MIMIC split CSV": "./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv",
    }
    if _re6:
        MIMIC["Retrained 6%"] = f"{_re6}/pytorch_model.bin"
    if _re10:
        MIMIC["Retrained 10%"] = f"{_re10}/pytorch_model.bin"
else:
    MIMIC = {}

# Detect IU datasets (optional)
iu_data_root = _find_kaggle_dataset('forget-mi-data-iu')
iu_models_root = _find_kaggle_dataset('forget-mi-models-iu')
if iu_data_root or iu_models_root:
    print(f"📦 forget-mi-data-iu   → {iu_data_root or '❌'}")
    print(f"📦 forget-mi-models-iu → {iu_models_root or '❌'}\n")
IU = {}
if iu_data_root and iu_models_root:
    IU = {
        "IU base model":   f"{iu_models_root}/base_model/model_og_IU/pytorch_model.bin",
        "IU retrained 3%": f"{iu_models_root}/retrained_model/model_retrained_iu_3per/pytorch_model.bin",
        "IU text data":    f"{iu_data_root}/data/metadata",
        "IU image data":   f"{iu_data_root}/data/img_data",
        "IU split CSV":    "./data_iu/splits/train_test_split_iu.csv",
        "IU forget 3%":    "./data_iu/splits/forget_set_3per_iu.csv",
    }


def _verify_group(title, checks, required=True):
    if not checks:
        print(f"━━ {title} ━━")
        icon = "❌" if required else "⚠️ "
        print(f"  {icon} Dataset gốc không tồn tại trong /kaggle/input/")
        return False
    print(f"━━ {title} ━━")
    all_ok = True
    for name, path in checks.items():
        if os.path.exists(path):
            if os.path.isdir(path):
                n = len(os.listdir(path))
                print(f"  ✅ {name:<18} {path}  ({n} items)")
            else:
                size = os.path.getsize(path) / 1e6
                print(f"  ✅ {name:<18} {path}  ({size:.1f} MB)")
        else:
            icon = "❌" if required else "⚠️ "
            print(f"  {icon} {name:<18} {path}  KHÔNG TỒN TẠI")
            if required:
                all_ok = False
    return all_ok


common_ok = _verify_group("FILES CHUNG (từ repo clone)", COMMON, required=True)
print()
mimic_ok = _verify_group("MIMIC-CXR (BẮT BUỘC cho Cell 4a/4b/4c)", MIMIC, required=True)
print()
iu_ok = _verify_group("INDIANA UNIVERSITY CXR (optional)", IU, required=False) if IU else False

# Export paths for Cell 3 helpers
KAGGLE_MIMIC_DATA_ROOT = mimic_data_root
KAGGLE_MIMIC_MODELS_ROOT = mimic_models_root
KAGGLE_IU_DATA_ROOT = iu_data_root
KAGGLE_IU_MODELS_ROOT = iu_models_root

print()
print("━" * 70)
if common_ok and mimic_ok:
    print("✅ MIMIC-CXR sẵn sàng — chạy được Cell 4a/4b/4c")
    print(f"   data_root  = {mimic_data_root}/data")
    print(f"   models_root= {mimic_models_root}")
else:
    if not mimic_data_root or not mimic_models_root:
        print("❌ Kaggle Datasets chưa add hoặc tên không khớp:")
        print("   1. Sidebar phải → + Add Data → search forget-mi-data → Add")
        print("   2. Sidebar phải → + Add Data → search forget-mi-models → Add")
        print("   3. Run → Restart Session → Rerun Cell 1 và Cell 2")
    else:
        print("❌ Datasets có nhưng cấu trúc bên trong không khớp")
        print("   Run cell diagnostic (find pytorch_model.bin) để xem nesting thực tế")
print()
if iu_data_root and iu_models_root:
    print("✅ IU-CXR sẵn sàng — Cell 3 có thể bật DATASETS['iu']['enabled']=True")
else:
    print("⚠️  IU-CXR chưa có Kaggle Datasets (bình thường nếu chưa prep)")

In [ ]:
# ====================================
# CELL 3: Helpers cho baseline multi-seed (BẮT BUỘC trước Cell 4*)
# ====================================
# Support 2 datasets: MIMIC-CXR + Indiana University CXR.
# DATASETS dict dùng AUTO-DETECTED paths từ Cell 2 (KAGGLE_*_ROOT vars).
# → Tự động handle cả 2 format Kaggle mount (cũ + mới /datasets/<user>/).

import os, glob, numpy as np, pandas as pd
from datetime import datetime

CSV_PATH = "/kaggle/working/results_summary.csv"

PAPER_REF_MIMIC = {
    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},
    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},
    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},
}


def _find_kaggle_dataset(slug):
    """Find Kaggle dataset by slug, handle cả 2 format mount."""
    direct = f'/kaggle/input/{slug}'
    if os.path.isdir(direct):
        return direct
    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')
    return candidates[0] if candidates else None


# Auto-detect paths (re-detect ở đây phòng khi Cell 2 không chạy)
_mimic_data = globals().get('KAGGLE_MIMIC_DATA_ROOT') or _find_kaggle_dataset('forget-mi-data')
_mimic_models = globals().get('KAGGLE_MIMIC_MODELS_ROOT') or _find_kaggle_dataset('forget-mi-models')
_iu_data = globals().get('KAGGLE_IU_DATA_ROOT') or _find_kaggle_dataset('forget-mi-data-iu')
_iu_models = globals().get('KAGGLE_IU_MODELS_ROOT') or _find_kaggle_dataset('forget-mi-models-iu')

# IU detection: CHỈ quét trong dataset IU (nếu được add). KHÔNG quét /kaggle/input/**
# — recursive glob toàn mount (gồm ~6700 ảnh MIMIC qua FUSE) làm Cell 3 chậm vài phút,
# và còn nhận nhầm all_data.tsv của MIMIC. IU chưa add → bỏ qua, _iu_* = None/''.
# og + re có thể ở 2 dataset riêng (forget-mi-models-iu + forget-mi-models-iu-re;
# cùng account không đặt trùng tên được) → gom MỌI dataset khớp prefix vào roots.
_iu_models_all = glob.glob('/kaggle/input/datasets/*/forget-mi-models-iu*') + glob.glob('/kaggle/input/forget-mi-models-iu*')
_iu_roots = list(dict.fromkeys([r for r in ([_iu_data, _iu_models] + _iu_models_all) if r]))

def _find_one(_name):
    for _r in _iu_roots:
        _h = glob.glob(f'{_r}/**/{_name}', recursive=True)
        if _h:
            return sorted(_h, key=len)[0]
    return None

_iu_meta_tsv = _find_one('all_data.tsv') if _iu_roots else None
_iu_meta_dir = os.path.dirname(_iu_meta_tsv) if _iu_meta_tsv else None
_iu_data_root = os.path.dirname(_iu_meta_dir) if _iu_meta_dir else (f'{_iu_data}/data' if _iu_data else None)
# Cách 3: ảnh IU không đóng gói → trỏ sang raddar (attach kèm). Nếu data_root có img_data (bản cũ) thì ưu tiên.
_iu_img = (os.path.join(_iu_data_root, 'img_data')
           if _iu_data_root and os.path.isdir(os.path.join(_iu_data_root, 'img_data')) else None)
if not _iu_img:
    for _r in glob.glob('/kaggle/input/*chest-xray*') + glob.glob('/kaggle/input/datasets/*/*chest-xray*'):
        for _sub in ('images/images_normalized', 'images/images', 'images_normalized', 'images'):
            _p = os.path.join(_r, _sub)
            if os.path.isdir(_p) and glob.glob(os.path.join(_p, '*.png')):
                _iu_img = _p; break
        if _iu_img: break
_iu_split = _find_one('iu-split.csv') if _iu_roots else None
_iu_f3 = _find_one('forget_set_3per_iu.csv') if _iu_roots else None
_iu_forget_dir = os.path.dirname(_iu_f3) if _iu_f3 else None
_iu_og_bin = _find_one('model_og_IU/pytorch_model.bin') if _iu_roots else None
_iu_og_dir = os.path.dirname(_iu_og_bin) if _iu_og_bin else ''
_iu_re3_bin = _find_one('model_retrained_iu_3per/pytorch_model.bin') if _iu_roots else None
_iu_re3_dir = os.path.dirname(_iu_re3_bin) if _iu_re3_bin else None

DATASETS = {
    'mimic': {
        'enabled': _mimic_data is not None and _mimic_models is not None,
        'name': 'MIMIC-CXR',
        'config_file': 'config_baseline_kaggle.yaml',
        # data_root = thư mục chứa metadata/, img_data/, text_data/ TRỰC TIẾP
        'data_root': f'{_mimic_data}/data' if _mimic_data else None,
        'models_root': _mimic_models,
        # 'forget-mi-models-full' nesting: original_model/forgetme/training_original_model/
        #                                  model_retrained_<N>per/model_retrained_<N>per/
        # Fallback paths cho 'forget-mi-models' old nesting:
        #   base_model/training_original_model/  +  retrained_model/model_retrained_3per/
        'base_model_subdir': 'original_model/forgetme/training_original_model' if (_mimic_models and 'forget-mi-models-full' in _mimic_models) else 'base_model/training_original_model',
        'gold_retrained': (
            {3:  'model_retrained_3per/model_retrained_3per',
             6:  'model_retrained_6per/model_retrained_6per',
             10: 'model_retrained_10per/model_retrained_10per'}
            if (_mimic_models and 'forget-mi-models-full' in _mimic_models)
            else {3: 'retrained_model/model_retrained_3per', 6: None, 10: None}
        ),
        'data_split': './data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv',
        'forget_set_template': './data_splits/forget_set_{}per.csv',
        'paper_ref': PAPER_REF_MIMIC,
        'extra_overrides': '',
    },
    'iu': {
        'enabled': bool(_iu_data_root and _iu_og_dir),
        'name': 'Indiana University CXR',
        'config_file': 'config_baseline_iu_kaggle.yaml',
        'data_root': _iu_data_root,
        'img_dir': _iu_img,
        # models_root='' → base_model_subdir / gold paths are ABSOLUTE (robust to nesting).
        'models_root': '',
        'base_model_subdir': _iu_og_dir,
        'gold_retrained': {3: _iu_re3_dir, 6: None, 10: None},
        'data_split': _iu_split or './data_iu/data_splits/iu-split.csv',
        'forget_set_template': (os.path.join(_iu_forget_dir, 'forget_set_{}per_iu.csv')
                                if _iu_forget_dir else './data_iu/data_splits/forget_set_{}per_iu.csv'),
        'paper_ref': None,
        'extra_overrides': 'output_channel_encoding=multiclass',
    },
}

METRIC_DEFS = [
    ('MIA',                 'MIA_persample',   None,         '↓'),
    ('MIA_paper',           'MIA_paper',       'MIA_paper',  '↓'),
    ('forget_ce',           'forget_ce',       None,         '·'),
    ('test_ce',             'test_ce',         None,         '·'),
    ('Df_AUC',              'Forget AUC',      'Df_AUC',     '↓'),
    ('Df_F1',               'Forget Mac-F1',   'Df_F1',      '↓'),
    ('Dt_AUC',              'Test AUC',        'Dt_AUC',     '↑'),
    ('Dt_F1',               'Test Mac-F1',     'Dt_F1',      '↑'),
    ('dist_vs_re',          '1 − CosSim',      None,         '↓'),
    ('unlearn_time_hours',  'Time (h)',        'Time_h',     '↓'),
    ('gpu_peak_GB',         'GPU peak (GB)',   None,         '·'),
    ('trainable_ratio',     'Trainable ratio', None,         '·'),
]


def _check_dataset_ready(dataset_key):
    """Verify dataset config + Kaggle input paths đã sẵn sàng."""
    ds = DATASETS[dataset_key]
    if not ds['enabled']:
        return False, f"DATASETS['{dataset_key}']['enabled']=False — Kaggle Dataset chưa add"
    if not ds.get('data_root') or not ds.get('models_root'):
        return False, f"data_root hoặc models_root = None — auto-detect fail"
    if not os.path.exists(ds['config_file']):
        return False, f"Config file không tồn tại: {ds['config_file']}"
    base_model = os.path.join(ds['models_root'], ds['base_model_subdir'])
    if not os.path.isdir(base_model):
        return False, f"Pretrained model không tồn tại: {base_model}/"
    if not os.path.exists(os.path.join(base_model, 'pytorch_model.bin')):
        return False, f"pytorch_model.bin không tồn tại trong {base_model}/"
    if not os.path.isdir(ds['data_root']):
        return False, f"Data root không tồn tại: {ds['data_root']}/"
    if not os.path.isdir(os.path.join(ds['data_root'], 'metadata')):
        return False, f"Không tìm thấy {ds['data_root']}/metadata/"
    _imgd = ds.get('img_dir') or os.path.join(ds['data_root'], 'img_data')
    if not os.path.isdir(_imgd):
        return False, f"Image dir không tồn tại: {_imgd}/"
    return True, "ready"


def _check_gold(dataset_key, forget_pct):
    ds = DATASETS[dataset_key]
    subdir = ds['gold_retrained'].get(forget_pct)
    if subdir is None or not ds.get('models_root'):
        return None, False
    path = os.path.join(ds['models_root'], subdir)
    return path, os.path.isdir(path)


def _seed_done(dataset_key, forget_pct, seed):
    if not os.path.exists(CSV_PATH):
        return False
    try:
        df = pd.read_csv(CSV_PATH)
    except Exception:
        return False
    for col in ('forget_pct', 'seed', 'method'):
        if col not in df.columns:
            return False
    mask = (df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
            & (df['seed'] == seed)
            & (df['method'] == 'baseline_partial'))
    if 'dataset' in df.columns:
        mask = mask & (df['dataset'] == dataset_key)
    return bool(mask.any())


def _push_progress(tag=""):
    """Push experiments/ (CSV + exp MD + INDEX) lên GitHub NGAY sau mỗi seed.
    Timeout-safe: seed nào xong là đã trên GitHub → session sau Cell 1 restore CSV
    và _seed_done() tự skip seed đó. KHÔNG cần chờ tới Cell cuối mới push."""
    import shutil, subprocess as _sp
    # 1. forgetmi_partial ghi CSV trực tiếp vào CSV_PATH → copy sang file repo mà Cell 1 restore
    if os.path.exists(CSV_PATH):
        os.makedirs("experiments", exist_ok=True)
        shutil.copy(CSV_PATH, "experiments/results_summary_kaggle.csv")
    # 2. Credentials từ Kaggle Secrets
    try:
        from kaggle_secrets import UserSecretsClient
        _sec = UserSecretsClient()
        _tok = _sec.get_secret("GITHUB_TOKEN")
        _email = _sec.get_secret("GIT_EMAIL")
        _name = _sec.get_secret("GIT_NAME")
    except Exception as _e:
        print(f"   ⚠️  Push-progress skip (no secrets): {_e}")
        return
    if not (_tok and _email and _name):
        print("   ⚠️  Push-progress skip — thiếu credentials")
        return
    _repo = "nhnhu146/Forget-MI-LoKU"
    _sp.run(["git", "config", "user.email", _email])
    _sp.run(["git", "config", "user.name", _name])
    _sp.run(["git", "remote", "set-url", "origin", f"https://{_tok}@github.com/{_repo}.git"])
    _sp.run(["git", "add", "experiments/"])
    _staged = _sp.run(["git", "diff", "--cached", "--name-only"],
                      capture_output=True, text=True).stdout.strip()
    if not _staged:
        print("   ℹ️  Push-progress: không có gì mới")
        return
    _sp.run(["git", "commit", "-m", f"baseline progress: {tag}".strip()])
    # rebase để hoà với commit khác, rồi push (retry 1 lần)
    _sp.run(["git", "pull", "--rebase", "--autostash", "origin", "master"],
            capture_output=True, text=True)
    _p = _sp.run(["git", "push", "origin", "master"], capture_output=True, text=True)
    if _p.returncode != 0:
        _sp.run(["git", "pull", "--rebase", "--autostash", "origin", "master"],
                capture_output=True, text=True)
        _p = _sp.run(["git", "push", "origin", "master"], capture_output=True, text=True)
    print(f"   {'✅ Pushed progress' if _p.returncode == 0 else '⚠️  Push failed (sẽ thử lại seed sau)'}: {tag}")


def run_baseline_multiseed(forget_pct, dataset='mimic', seeds=(42, 123, 7), force_redo=False):
    """Chạy baseline forgetmi_partial.py multi-seed."""
    # Normalize seeds: accept int / single-paren (42) / tuple / list
    if isinstance(seeds, int):
        seeds = (seeds,)
    seeds = tuple(seeds)
    assert forget_pct in (3, 6, 10)
    assert dataset in DATASETS

    ds = DATASETS[dataset]
    ready, msg = _check_dataset_ready(dataset)
    if not ready:
        print(f"❌ Dataset '{dataset}' chưa sẵn sàng: {msg}")
        return

    forget_csv = ds['forget_set_template'].format(forget_pct)
    retrained_path, has_gold = _check_gold(dataset, forget_pct)
    exp_name = f"baseline_{dataset}_{forget_pct}per"
    paper_ref = ds['paper_ref'][forget_pct] if ds['paper_ref'] else None
    base_model = os.path.join(ds['models_root'], ds['base_model_subdir'])

    print(f"\n{'#'*78}")
    print(f"# 🎯 BASELINE Forget-MI — {ds['name']} — FORGET {forget_pct}% — seeds {list(seeds)}")
    print(f"# Config         : {ds['config_file']}")
    print(f"# Data root      : {ds['data_root']}")
    print(f"# Base model     : {base_model}")
    print(f"# Forget CSV     : {forget_csv}")
    if has_gold:
        print(f"# Gold retrained : ✅ {retrained_path}")
    else:
        print(f"# Gold retrained : ❌ N/A → 1−CosSim KHÔNG hợp lệ")
    print(f"# ⏱  Dự kiến     : ~{5 * len(seeds)}h tổng")
    print(f"{'#'*78}\n")

    if not os.path.exists(forget_csv):
        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")

    OVR_parts = [
        f"forget_set_path={forget_csv}",
        f"id={exp_name}",
        f"base_model_path={base_model}",
        f"bert_pretrained_dir={base_model}",
        f"text_data_dir={ds['data_root']}/metadata",
        f"img_data_dir={ds.get('img_dir') or ds['data_root'] + '/img_data'}",
        f"data_split_path={ds['data_split']}",
        "output_dir=/kaggle/working/baseline_output",
    ]
    if has_gold:
        OVR_parts.append(f"retrained_model_path={retrained_path}")
    else:
        # No gold for this forget% → fallback to 3% model so script doesn't crash on load.
        # CosSim vs 3per is meaningless but eval helper will mark dist_vs_re as ⚠️ invalid.
        fallback = os.path.join(ds['models_root'], ds['gold_retrained'].get(3, '') or '')
        if fallback and os.path.isdir(fallback):
            print(f"#   (fallback retrained_model_path → {fallback} so script can load)")
            OVR_parts.append(f"retrained_model_path={fallback}")
    if ds['extra_overrides']:
        OVR_parts.append(ds['extra_overrides'])
    # Eval mỗi epoch (READ-ONLY) → ghi /kaggle/working/perepoch_<id>.csv để dò epoch khớp paper.
    # Kết quả epoch CUỐI (E29) KHÔNG đổi vì eval khôi phục RNG. Tắt: đặt EVAL_EVERY_EPOCH=False.
    if globals().get('EVAL_EVERY_EPOCH', True):
        OVR_parts.append("eval_every_epoch=1")
    OVR = ",".join(OVR_parts)

    HYPOTHESIS = (f"Baseline Forget-MI tại {ds['name']} {forget_pct}%, Kaggle reproduce.")

    for i, s in enumerate(seeds):
        if not force_redo and _seed_done(dataset, forget_pct, s):
            print(f"⏭️  SEED {s} đã có — bỏ qua\n")
            continue
        print(f"\n{'='*60}\n🎲 SEED {s} ({i+1}/{len(seeds)}) ▸ {dataset.upper()} {forget_pct}%\n{'='*60}")
        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_partial.py '
               f'--config {ds["config_file"]} --fresh --seed {s} '
               f'--override "{OVR}" '
               f'--exp {exp_name}_seed{s} --hypothesis "{HYPOTHESIS}"')
        get_ipython().system(cmd)
        _push_progress(f"{dataset} {forget_pct}% seed {s}")

    src_csv = "/kaggle/working/baseline_output/results_summary.csv"
    if os.path.exists(src_csv):
        df_src = pd.read_csv(src_csv)
        if 'dataset' not in df_src.columns:
            df_src['dataset'] = dataset
        if os.path.exists(CSV_PATH):
            df_main = pd.read_csv(CSV_PATH)
            df_merged = pd.concat([df_main, df_src], ignore_index=True)
            df_merged = df_merged.drop_duplicates(
                subset=['dataset', 'forget_pct', 'seed'], keep='last'
            ) if 'dataset' in df_merged.columns else df_merged
            df_merged.to_csv(CSV_PATH, index=False)
        else:
            df_src.to_csv(CSV_PATH, index=False)
        os.remove(src_csv)

    aggregate_baseline_summary(dataset, forget_pct, seeds, exp_name, paper_ref, has_gold)


def aggregate_baseline_summary(dataset_key, forget_pct, seeds, exp_name, paper_ref, has_gold):
    """In bảng mean±std + so paper + lưu MD summary."""
    if not os.path.exists(CSV_PATH):
        print(f"❌ {CSV_PATH} không tồn tại")
        return
    df = pd.read_csv(CSV_PATH)
    mask = (df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per")
            & df['seed'].isin(seeds))
    if 'method' in df.columns:
        mask = mask & (df['method'] == 'baseline_partial')
    if 'dataset' in df.columns:
        mask = mask & (df['dataset'] == dataset_key)
    df = df[mask]
    if df.empty:
        print(f"❌ Không có rows cho {dataset_key} {forget_pct}%")
        return
    if 'timestamp' in df.columns:
        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')

    ds = DATASETS[dataset_key]
    print(f"\n{'='*100}\n📊 BASELINE — {ds['name']} — FORGET {forget_pct}% — seeds={list(seeds)}\n{'='*100}")
    hdr = (f"{'Metric':<20}" + "".join(f"{f'seed{s}':>10}" for s in seeds)
           + f"{'mean ± std':>18}")
    if paper_ref:
        hdr += f"{'paper':>10}{'Δ vs paper':>12}"
    print(hdr); print("-" * len(hdr))

    md_cols = ["Metric"] + [f"seed {s}" for s in seeds] + ["**mean ± std**"]
    if paper_ref:
        md_cols += ["Paper", "Δ vs paper"]
    md_rows = ["| " + " | ".join(md_cols) + " |", "|" + "---|" * len(md_cols)]

    for csv_key, label, paper_key, direction in METRIC_DEFS:
        if csv_key not in df.columns:
            continue
        vals = []
        for s in seeds:
            sub = df[df['seed'] == s]
            if sub.empty: continue
            try:
                v = float(sub[csv_key].iloc[-1])
            except Exception:
                continue
            if v != v: continue
            vals.append(v)
        if not vals: continue
        m, sd = float(np.mean(vals)), float(np.std(vals))
        invalid_marker = " ⚠️" if (csv_key == 'dist_vs_re' and not has_gold) else ""
        if paper_ref and paper_key and paper_key in paper_ref:
            p_val = paper_ref[paper_key]
            delta = m - p_val
            good = (direction == '↓' and delta < 0) or (direction == '↑' and delta > 0)
            sym = "✅" if good else ("❌" if direction in ('↓','↑') else "·")
            paper_console = f"{p_val:>10.3f}{sym}{delta:+.3f}"
            md_paper, md_delta = f"{p_val:.3f}", f"{delta:+.3f}"
        else:
            paper_console = ""
            md_paper, md_delta = "—", "—"
        cell_vals = "".join(f"{v:>10.3f}" for v in vals) + " " * (10 * (len(seeds) - len(vals)))
        print(f"{label+invalid_marker:<20}{cell_vals}{m:>10.3f}±{sd:.3f}{paper_console}")
        md_vals_str = " | ".join(f"{v:.3f}" for v in vals) + " | " * (len(seeds) - len(vals))
        md_row = f"| {label}{invalid_marker} | {md_vals_str} | **{m:.3f} ± {sd:.3f}** |"
        if paper_ref:
            md_row += f" {md_paper} | {md_delta} |"
        md_rows.append(md_row)

    os.makedirs("experiments", exist_ok=True)
    out_md = f"experiments/summary_baseline_{dataset_key}_{forget_pct}per_multiseed.md"
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join([
            f"# BASELINE Forget-MI — {ds['name']} — FORGET {forget_pct}%",
            f"\n_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} on Kaggle_\n",
            f"**Dataset**: {ds['name']} (`{dataset_key}`)",
            f"**Seeds**: {list(seeds)}",
            f"**Gold retrain**: {'✅' if has_gold else '❌ N/A'}\n",
            *md_rows,
        ]))
    print(f"\n💾 Summary MD: {out_md}")


# ---- Print readiness status ----
print("✅ Helpers loaded.\n")
print("📋 Dataset readiness (auto-detected paths):")
for k, ds in DATASETS.items():
    status = "✅ ENABLED" if ds['enabled'] else "❌ DISABLED"
    print(f"   • {k:<6} ({ds['name']}): {status}")
    if ds.get('data_root'):
        print(f"            data_root : {ds['data_root']}")
    if ds.get('models_root'):
        print(f"            base_model: {os.path.join(ds['models_root'], ds['base_model_subdir'])}")
    if ds['enabled']:
        ok, msg = _check_dataset_ready(k)
        print(f"            {'✅ ready' if ok else '⚠️  ' + msg}")
print(f"\nUsage: run_baseline_multiseed(forget_pct=3, dataset='mimic')")

---

## 🎯 Training cells — chạy theo nhu cầu

⚠️ **Quota Kaggle free: 30h GPU/tuần**. Mỗi forget% × 3 seeds ≈ 15h.

Khuyến nghị thứ tự (theo priority luận văn):
1. **Cell 4a (MIMIC 3%)** — quan trọng nhất (so paper Table 2 Unimodal 3%)
2. **Cell 4c (MIMIC 10%)** — case khó nhất paper
3. **Cell 4b (MIMIC 6%)** — case trung gian
4. **Cell 4d-4f (IU 3/6/10%)** — generalization check (chỉ khi IU prep xong)

### 📊 MIMIC-CXR — Cells 4a, 4b, 4c

In [ ]:
# ====================================
# CELL 4a: MIMIC-CXR — FORGET 3% (multi-seed)
# ====================================
# Expected: paper Table 2 → MIA=0.571, Df_AUC=0.735, Dt_AUC=0.625, Time~5h
# ⚙️ Toggle RUN_3PER = True để chạy cell này.
# Helper auto-skip seeds đã có trong CSV restored từ session trước.

RUN_3PER = True                    # ⚙️ Set True để chạy 3%
FORCE_REDO = True                  # ⚙️ True = chạy LẠI dù CSV đã có kết quả (bỏ qua _seed_done skip).
                                   #    Dùng khi re-run để lấy perepoch. Xong đặt False để khỏi chạy lại.
SEEDS_THIS_RUN = (42,)            # 1 seed/% (code gốc paper, weights 1/1/2/2)

if RUN_3PER:
    run_baseline_multiseed(
        forget_pct=3,
        dataset='mimic',
        seeds=SEEDS_THIS_RUN,
        force_redo=FORCE_REDO,
    )
else:
    print("⏭️  Skip MIMIC 3% (RUN_3PER=False)")


In [ ]:
# ====================================
# CELL 4b: MIMIC-CXR — FORGET 6% (multi-seed)
# ====================================
# Expected: paper Table 2 → MIA=0.615, Df_AUC=0.654, Dt_AUC=0.599
# ⚙️ Toggle RUN_6PER = True để chạy cell này.

RUN_6PER = True                    # ⚙️ Set True để chạy 6%
SEEDS_THIS_RUN = (42,)            # 1 seed/% (code gốc paper, weights 1/1/2/2)      # ⚙️ Sửa nếu muốn chạy 1 seed: (42,)

if RUN_6PER:
    run_baseline_multiseed(
        forget_pct=6,
        dataset='mimic',
        seeds=SEEDS_THIS_RUN,
    )
else:
    print("⏭️  Skip MIMIC 6% (RUN_6PER=False)")


In [ ]:
# ====================================
# CELL 4c: MIMIC-CXR — FORGET 10% (multi-seed)
# ====================================
# Expected: paper Table 2 → MIA=0.810, Df_AUC=0.656, Dt_AUC=0.565
# ⚙️ Toggle RUN_10PER = True để chạy cell này.

RUN_10PER = True                   # ⚙️ Set True để chạy 10%
SEEDS_THIS_RUN = (42,)            # 1 seed/% (code gốc paper, weights 1/1/2/2)      # ⚙️ Sửa nếu muốn chạy 1 seed: (42,)

if RUN_10PER:
    run_baseline_multiseed(
        forget_pct=10,
        dataset='mimic',
        seeds=SEEDS_THIS_RUN,
    )
else:
    print("⏭️  Skip MIMIC 10% (RUN_10PER=False)")


---

## 🔬 Indiana University CXR — generalization check

⚠️ **Yêu cầu trước khi chạy cells dưới**:
1. Preprocess IU-CXR theo Section 13 của `THESIS_ROADMAP.md` (download Open-i, parse XML, sinh label binary normal/abnormal, tạo splits, retrain models)
2. Tạo file `config_baseline_iu_kaggle.yaml` (copy `config_baseline_kaggle.yaml`, đổi `output_channel_encoding=binary` + paths IU)
3. Upload 2 Kaggle Datasets:
   - `forget-mi-data-iu` (metadata + img_data IU)
   - `forget-mi-models-iu` (model_og_IU + model_retrained_iu_*per)
4. Set `DATASETS['iu']['enabled'] = True` trong Cell 3 → rerun Cell 3 → chạy cells dưới

Nếu chưa làm xong, cells này sẽ chỉ in thông báo "dataset chưa sẵn sàng" và skip — không crash notebook.

In [ ]:
# ====================================
# CELL 4d: IU-CXR — FORGET 3% (multi-seed)
# ====================================
# Task: binary normal/abnormal. ⚙️ Cần upload Kaggle Dataset 'forget-mi-data-iu'
# + 'forget-mi-models-iu' trước. Cell auto-skip nếu DATASETS['iu']['enabled']=False.

RUN_IU_3PER = False                # ⚙️ Set True khi IU dataset đã sẵn
SEEDS_THIS_RUN = (42, 123, 7)

if RUN_IU_3PER:
    run_baseline_multiseed(
        forget_pct=3,
        dataset='iu',
        seeds=SEEDS_THIS_RUN,
    )
else:
    print("⏭️  Skip IU 3% (RUN_IU_3PER=False)")


In [ ]:
# ====================================
# CELL 4e: IU-CXR — FORGET 6% (multi-seed)
# ====================================
RUN_IU_6PER = False                # ⚙️ Set True khi cần
SEEDS_THIS_RUN = (42, 123, 7)

if RUN_IU_6PER:
    run_baseline_multiseed(
        forget_pct=6,
        dataset='iu',
        seeds=SEEDS_THIS_RUN,
    )
else:
    print("⏭️  Skip IU 6% (RUN_IU_6PER=False)")


In [ ]:
# ====================================
# CELL 4f: IU-CXR — FORGET 10% (multi-seed)
# ====================================
RUN_IU_10PER = False               # ⚙️ Set True khi cần
SEEDS_THIS_RUN = (42, 123, 7)

if RUN_IU_10PER:
    run_baseline_multiseed(
        forget_pct=10,
        dataset='iu',
        seeds=SEEDS_THIS_RUN,
    )
else:
    print("⏭️  Skip IU 10% (RUN_IU_10PER=False)")


In [ ]:
# ====================================
# CELL 5: BẢNG CROSS-DATASET — Baseline trên MIMIC + IU (cho Chương 4 luận văn)
# ====================================
# Đọc CSV chung → in bảng so sánh 2 datasets × 3 forget% × Paper ref (chỉ MIMIC).
# Chạy sau khi đã hoàn thành ≥ 1 cell 4* (MIMIC hoặc IU).

import os, numpy as np, pandas as pd
from datetime import datetime

if not os.path.exists(CSV_PATH):
    print(f"❌ {CSV_PATH} không tồn tại. Chạy ít nhất 1 cell 4* trước.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"📋 Tổng rows: {len(df)}")
    for col in ('dataset', 'method', 'forget_pct'):
        if col in df.columns:
            print(f"📋 {col:<12}: {df[col].unique().tolist()}")
    print()

    show_metrics = [
        ('MIA_paper',          'MIA',         '↓'),
        ('Df_AUC',             'Df_AUC',      '↓'),
        ('Df_F1',              'Df_F1',       '↓'),
        ('Dt_AUC',             'Dt_AUC',      '↑'),
        ('Dt_F1',              'Dt_F1',       '↑'),
        ('dist_vs_re',         '1−CosSim',    '↓'),
        ('unlearn_time_hours', 'Time(h)',     '↓'),
    ]

    md = [
        "# Bảng baseline Kaggle — Cross-dataset (MIMIC + IU)",
        "",
        f"_Auto-generated từ `{CSV_PATH}` — {datetime.now().strftime('%Y-%m-%d %H:%M')}_",
        "",
        "Bảng chứa BASELINE Forget-MI trên Kaggle cho cả 2 datasets.",
        "Để so với LoKU, merge CSV này với CSV từ Colab.",
        "",
    ]

    header = "| Dataset | Forget% | Method | " + " | ".join(m[1] for m in show_metrics) + " |"
    sep = "|---|---|---|" + "---|" * len(show_metrics)
    md += [header, sep]
    print("=" * 115); print(header); print("=" * 115)

    paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',
                     'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}

    for dataset_key in ['mimic', 'iu']:
        ds = DATASETS[dataset_key]
        # Filter rows for this dataset
        if 'dataset' in df.columns:
            sub_ds = df[df['dataset'] == dataset_key]
        else:
            # Fallback: cũ không có column dataset → assume mimic
            sub_ds = df if dataset_key == 'mimic' else df.iloc[0:0]

        if sub_ds.empty and not ds['enabled']:
            # IU chưa enable + chưa có data → skip section
            continue

        # Section divider
        section_row = f"| **{ds['name']}** | | | " + " | ".join("" for _ in show_metrics) + " |"
        md.append(section_row)
        print(f"\n━━━ {ds['name']} ━━━")

        for pct in [3, 6, 10]:
            sub_pct = sub_ds[sub_ds['forget_pct'].astype(str).str.contains(f"_{pct}per")] if not sub_ds.empty else sub_ds
            if 'method' in sub_pct.columns:
                sub_pct = sub_pct[sub_pct['method'] == 'baseline_partial']

            # ----- Paper row (chỉ MIMIC) -----
            if ds['paper_ref'] and pct in ds['paper_ref']:
                paper_row = f"| {ds['name']} | {pct}% | Paper |"
                for csv_k, _, _ in show_metrics:
                    if csv_k in paper_key_map and paper_key_map[csv_k] in ds['paper_ref'][pct]:
                        paper_row += f" {ds['paper_ref'][pct][paper_key_map[csv_k]]:.3f} |"
                    else:
                        paper_row += " — |"
                md.append(paper_row); print(paper_row)

            # ----- Baseline row -----
            if sub_pct.empty:
                row = f"| {ds['name']} | {pct}% | _(chưa chạy)_ |" + " — |" * len(show_metrics)
            else:
                if 'timestamp' in sub_pct.columns:
                    sub_pct = sub_pct.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')
                n = sub_pct['seed'].nunique()
                row = f"| {ds['name']} | {pct}% | **Baseline (n={n})** |"
                for csv_k, _, _ in show_metrics:
                    if csv_k not in sub_pct.columns:
                        row += " — |"; continue
                    vals = sub_pct[csv_k].dropna().values
                    if len(vals) == 0:
                        row += " — |"; continue
                    mv, sd = float(np.mean(vals)), float(np.std(vals))
                    # mark CosSim invalid khi no gold
                    no_gold_pct = ds['gold_retrained'].get(pct) is None
                    mark = "⚠️" if (csv_k == 'dist_vs_re' and no_gold_pct) else ""
                    row += f" {mv:.3f}±{sd:.3f}{mark} |"
            md.append(row); print(row)

            # ----- Δ row (chỉ khi có paper + baseline) -----
            if (ds['paper_ref'] and pct in ds['paper_ref']
                    and not sub_pct.empty):
                delta_row = f"| {ds['name']} | {pct}% | Δ (LoKU−paper) |"
                for csv_k, _, direction in show_metrics:
                    if (csv_k not in sub_pct.columns
                            or csv_k not in paper_key_map
                            or paper_key_map[csv_k] not in ds['paper_ref'][pct]):
                        delta_row += " — |"; continue
                    vals = sub_pct[csv_k].dropna().values
                    if len(vals) == 0:
                        delta_row += " — |"; continue
                    mv = float(np.mean(vals))
                    p = ds['paper_ref'][pct][paper_key_map[csv_k]]
                    d = mv - p
                    good = (direction == '↓' and d < 0) or (direction == '↑' and d > 0)
                    sym = "✅" if good else "❌"
                    delta_row += f" {sym}{d:+.3f} |"
                md.append(delta_row); print(delta_row)
            md.append("")  # spacing
            print("-" * 115)

    out_md = "experiments/bang_baseline_kaggle_cross_dataset.md"
    with open(out_md, "w", encoding="utf-8") as f:
        f.write("\n".join(md))
    print(f"\n💾 Bảng cross-dataset saved: {out_md}")
    print(f"   → Dùng cho Chương 4.9-4.11 luận văn (so sánh MIMIC vs IU)")

In [ ]:
# ====================================
# CELL 6: Push results lên GitHub (Kaggle Secrets)
# ====================================
# Cần setup 1 lần: Add-ons → Secrets → Add secret:
#   GITHUB_TOKEN = ghp_xxx (scope: repo)
#   GIT_EMAIL    = your@email.com
#   GIT_NAME     = Your Name
#
# Cell này push:
#   1. experiments/*.md (tracker MD files + summaries)
#   2. experiments/results_summary_kaggle.csv (copy từ /kaggle/working/ để khỏi mất)
#      → CSV cần thiết cho _seed_done() check skip ở session sau

import os, shutil, subprocess

GITHUB_REPO = "nhnhu146/Forget-MI-LoKU"
BRANCH = "master"

# ----- BƯỚC 1: Copy CSV từ /kaggle/working/ → experiments/ để đưa vào git -----
CSV_SRC = "/kaggle/working/results_summary.csv"
CSV_DST = "experiments/results_summary_kaggle.csv"
if os.path.exists(CSV_SRC):
    os.makedirs("experiments", exist_ok=True)
    shutil.copy(CSV_SRC, CSV_DST)
    n_lines = sum(1 for _ in open(CSV_DST))
    print(f"📋 Copied CSV ({n_lines-1} rows) → {CSV_DST}")
else:
    print(f"ℹ️  CSV chưa có ({CSV_SRC}) — chỉ push MD files")

# ----- BƯỚC 2: Load credentials -----
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        return (secrets.get_secret('GITHUB_TOKEN'),
                secrets.get_secret('GIT_EMAIL'),
                secrets.get_secret('GIT_NAME'))
    except Exception as e:
        print(f"⚠️  Không load được Kaggle Secrets ({e})")
        return None, None, None

TOKEN, EMAIL, NAME = load_secrets()
if TOKEN and EMAIL and NAME:
    print("🔑 Credentials từ Kaggle Secrets ✅")
else:
    print("⚠️  Thiếu credentials — skip push")
    print("   Setup: Add-ons → Secrets → GITHUB_TOKEN + GIT_EMAIL + GIT_NAME")
    print(f"\n💾 Tải kết quả thủ công:")
    print(f"   1. {CSV_DST} (sweep state)")
    print(f"   2. experiments/*.md (tracker + summary)")
    print(f"   3. Hoặc: Save Version → Save Output để giữ /kaggle/working/")
    raise SystemExit

# ----- BƯỚC 3: Git commit + push -----
!git config user.email "{EMAIL}"
!git config user.name "{NAME}"
!git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git
!git add experiments/ 2>/dev/null

changes = !git diff --cached --name-only
if changes and any(c.strip() for c in changes):
    print("\n📦 Files commit:")
    for f in changes:
        if f.strip():
            print(f"   - {f}")
    msg = "baseline kaggle: auto-tracked multi-seed results + CSV checkpoint"
    !git commit -m "{msg}"
    !git push origin {BRANCH}
    print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")
    print(f"\n💡 SESSION SAU:")
    print(f"   - Cell 1 pull code → tự động kéo CSV về experiments/results_summary_kaggle.csv")
    print(f"   - Notebook tự đọc CSV này → _seed_done() skip đúng seeds đã xong")
else:
    print("ℹ️  Không có file mới để commit.")